# Query the Last Validated Served Face Recognition Model

The validated ONNX model version is deployed on **RHOAI** via **KServe Standard mode** with **OpenVINO Model Server (OVMS)**.

This notebook is the standard demo path. It does **not** retrain, upload, or restart the model server.
It sends images to the model server over HTTP and receives bounding-box predictions back through the
**KServe v2 REST API** — the same production inference pattern an application would use.


In [ ]:
!pip install --no-cache-dir -q -r requirements.txt

import cv2
import numpy as np
from matplotlib import pyplot as plt
from pathlib import Path
from IPython.display import Video, display
import remote_infer

## Configure the inference endpoint

KServe exposes each `InferenceService` as a Kubernetes Service with a predictable URL pattern:

```
http://<isvc-name>-predictor.<namespace>.svc.cluster.local:<port>/v2/models/<model-name>/infer
```

The workbench sets `FACE_RECOGNITION_ENDPOINT` so this notebook can run unchanged in the demo environment.
If that environment variable is absent, the notebook falls back to the standard in-cluster service URL.


In [ ]:
import json
import os
import subprocess
import requests
import boto3
from botocore.config import Config

MODEL_NAME = os.getenv("FACE_RECOGNITION_MODEL_NAME", "face-recognition")
NAMESPACE = os.getenv("FACE_RECOGNITION_NAMESPACE", "enterprise-mlops")
BASE_URL = os.getenv(
    "FACE_RECOGNITION_ENDPOINT",
    f"http://{MODEL_NAME}-predictor.{NAMESPACE}.svc.cluster.local:8888",
).rstrip("/")
INFER_URL = f"{BASE_URL}/v2/models/{MODEL_NAME}/infer"

print(f"InferenceService: {NAMESPACE}/{MODEL_NAME}")
print(f"Endpoint: {INFER_URL}")

# Show Model Registry linkage when the workbench service account can read the InferenceService.
try:
    raw = subprocess.check_output(
        ["oc", "get", "inferenceservice", MODEL_NAME, "-n", NAMESPACE, "-o", "json"],
        text=True,
        timeout=10,
    )
    isvc = json.loads(raw)
    labels = isvc.get("metadata", {}).get("labels", {})
    print("Model Registry linkage:")
    print(f"  registered-model-id: {labels.get('modelregistry.opendatahub.io/registered-model-id', 'not set')}")
    print(f"  model-version-id: {labels.get('modelregistry.opendatahub.io/model-version-id', 'not set')}")
    print(f"  storageUri: {isvc.get('spec', {}).get('predictor', {}).get('model', {}).get('storageUri', 'unknown')}")
except Exception as e:
    print(f"Could not read InferenceService metadata from the workbench: {e}")

# Show the served model artifact when MinIO is reachable from the workbench.
try:
    bucket = os.getenv("FACE_RECOGNITION_BUCKET", "models")
    key = os.getenv("FACE_RECOGNITION_MODEL_KEY", "face-recognition/1/model.onnx")
    s3 = boto3.client(
        "s3",
        endpoint_url=os.getenv("MINIO_ENDPOINT", "http://minio.minio-storage.svc:9000"),
        aws_access_key_id=os.getenv("MINIO_ACCESS_KEY", "rhoai-access-key"),
        aws_secret_access_key=os.getenv("MINIO_SECRET_KEY", "rhoai-secret-key-12345"),
        config=Config(signature_version="s3v4"),
    )
    obj = s3.head_object(Bucket=bucket, Key=key)
    size_mb = obj["ContentLength"] / (1024 * 1024)
    print(f"Served artifact: s3://{bucket}/{key}")
    print(f"  last_modified: {obj['LastModified']}")
    print(f"  size: {size_mb:.1f} MB")
except Exception as e:
    print(f"Could not read served model artifact metadata from MinIO: {e}")

# Readiness and metadata check
try:
    health = requests.get(f"{BASE_URL}/v2/health/ready", timeout=5)
    meta = requests.get(f"{BASE_URL}/v2/models/{MODEL_NAME}", timeout=5).json()
    print(f"Model server: READY (HTTP {health.status_code})")
    print(f"Model: {meta['name']}, platform: {meta.get('platform', 'unknown')}")
    for inp in meta.get("inputs", []):
        print(f"  Input: {inp['name']} shape={inp['shape']} dtype={inp['datatype']}")
    for out in meta.get("outputs", []):
        print(f"  Output: {out['name']} shape={out['shape']} dtype={out['datatype']}")
except Exception as e:
    print(f"Could not reach model server: {e}")
    print("Ensure the InferenceService is deployed and Ready:")
    print(f"  oc get inferenceservice {MODEL_NAME} -n {NAMESPACE}")


In [ ]:
# ============================================================
# INFERENCE CONFIGURATION — tweak these to tune demo sensitivity
# ============================================================
CONF_THRESHOLD = float(os.getenv("FACE_RECOGNITION_CONFIDENCE_THRESHOLD", "0.6"))
IDENTITY_DEDUP = True  # Apply one-adnan-per-frame constraint

print(f"Inference config: conf={CONF_THRESHOLD}, dedup={IDENTITY_DEDUP}")


## Single image inference via REST API

In [ ]:
import glob

test_images = sorted(glob.glob("images/test_face_*.jpg")) + sorted(glob.glob("images/test_group_*.jpg"))
print(f"Testing {len(test_images)} images via REST API\n")

for image_path in test_images:
    annotated_img, detections = remote_infer.process_image(image_path, INFER_URL, conf_threshold=CONF_THRESHOLD)

    print(f"--- {image_path} ---")
    for det in detections:
        print(f"  Detected: {det['class_name']} (confidence: {det['confidence']:.2f})")

    img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
    fig = plt.figure(figsize=(8, 5))
    plt.axis("off")
    plt.imshow(img_rgb)
    plt.title(f"{Path(image_path).name} — via Model Server")
    plt.show()

## Video inference via REST API

This demonstrates the **production inference pattern**: a client application extracts frames from a video stream,
sends each frame to the model server over HTTP, and reassembles the annotated results.

This is slower than local inference (network round-trip per frame) but scales independently —
the model server can be scaled horizontally to handle many concurrent clients.

In [ ]:
videos = ["videos/test_group_video.mov", "videos/test_face_video.mov", "videos/test_group_video_02.mov"]

for video_path in videos:
    if not Path(video_path).exists():
        print(f"Video not found: {video_path}")
        continue
    print(f"Processing {video_path} via REST API...")
    output_path = remote_infer.process_video_rest(video_path, INFER_URL, conf_threshold=CONF_THRESHOLD)
    import base64
    from IPython.display import HTML
    with open(output_path, "rb") as vf:
        video_b64 = base64.b64encode(vf.read()).decode()
    display(HTML(f'<video controls style="max-width:480px; border-radius:8px; box-shadow:0 2px 8px rgba(0,0,0,0.3)"><source src="data:video/webm;base64,{video_b64}" type="video/webm"></video>'))
    print()


## Conclusion

This notebook validated the production inference path without retraining or replacing the served model:

1. Used the RHOAI workbench as the client environment
2. Confirmed the KServe/OpenVINO model endpoint is reachable
3. Displayed Model Registry linkage when available on the `InferenceService`
4. Sent images and video frames through the KServe v2 REST API
5. Applied the same post-processing used by the demo application

This demonstrates **RHOAI 3.4's** ability to serve **predictive AI** computer-vision workloads with the same platform patterns used for generative AI: GitOps deployment, governed model versions, and production REST inference.
